In [ ]:
import os, shutil, random
import pydicom, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon
from pathlib import Path
from PIL import Image, ImageDraw
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
# source paths
WORK_DIR = Path("/home/jupyter-yin10/Image_Analysis")
IMG_DIR = Path("/data0/NIH-CXR14/images")
RAW_CSV = Path("/data0/NIH-CXR14/Data_Entry_2017_v2020.csv")

#### Random Circle r = 1, random intensity based on intensity histogram

In [ ]:
# output files
SAMPLE_CSV = WORK_DIR / "meta_NIH-CXR14.csv"
SAMPLE_WITH_PATHS = WORK_DIR / "NIH-CXR14_with_paths.csv"
out_root = WORK_DIR / "random_c1"

In [ ]:
# general settings
SEED = 42
dot_int = 255
WORK_DIR.mkdir(parents=True, exist_ok=True)

# setting circle radius
circle_radius = 1

In [ ]:
# load meta data
df_meta = pd.read_csv(RAW_CSV)
print("Meta data shape: {}".format(df_meta.shape))
print("\n")
print("Colums: {}".format(list(df_meta.columns)))
print("\n")
df_meta.to_csv(SAMPLE_CSV, index=False)
df_meta.head()

In [ ]:
# add image index column
df = pd.read_csv("/home/jupyter-yin10/Image_Analysis/meta_NIH-CXR14.csv")
df["Image Path"] = "/data0/NIH-CXR14/images/" + df["Image Index"]
df.to_csv("/home/jupyter-yin10/Image_Analysis/meta_NIH-CXR14_with_paths.csv")
print("Dataset shape: {}".format(df.shape))
print("\n")
print(df.head())

In [ ]:
# helper functions
def _seed_from_name(name: str, base_seed: int) -> int:
    #Stable per-image seed from base SEED and a stable string
    h = hashlib.sha256()
    h.update(str(base_seed).encode("utf-8"))
    h.update(b"::")
    h.update(name.encode("utf-8"))
    return int.from_bytes(h.digest()[:8], "little", signed=False)

def choose_random_center(img_path: Path, radius: int, base_seed: int, key_str: str) -> tuple[int, int]:
    with Image.open(img_path) as im:
        w, h = im.size  # (width, height)
    xmin, xmax = radius, max(radius, w - 1 - radius)
    ymin, ymax = radius, max(radius, h - 1 - radius)
    if xmax < xmin or ymax < ymin:
        # too small; fall back to center
        return (w // 2, h // 2)
    s = _seed_from_name(key_str, base_seed)
    rng = random.Random(s)
    cx = rng.randint(xmin, xmax)
    cy = rng.randint(ymin, ymax)
    return (cx, cy)

def compute_intensity(img_path: Path, base_seed: int, key_str: str) -> int:
    # get historgram peak x in 0 to 255
    # set range to x-20 to x+20
    # pick a random value for intensity
    with Image.open(img_path) as img:
        img = img.convert("L")
        hist = img.histogram()[:256]
    x = int(np.argmax(hist))
    lo = max(0, x-20)
    hi = min(255, x+20)
    rs = _seed_from_name("intensity::" + key_str, base_seed)
    rng = random.Random(rs)
    return int(rng.randint(int(lo),int(hi)))


def add_circle(img_path: Path, out_path: Path, center: tuple[int,int], radius:int, intensity:int):
    with Image.open(img_path) as img:
        img = img.convert("L")
        draw = ImageDraw.Draw(img)
        cx, cy = center
        r = int(radius)
        if r > 0:
            draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=int(intensity))
        out_path.parent.mkdir(parents=True,exist_ok=True)
        img.save(out_path)
        
def copy_or_link(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(src, dst)
    except OSError:
        try:
            # Symlink if allowed
            os.symlink(src, dst)
        except OSError:
            # Fallback to plain file copy 
            shutil.copyfile(src, dst)
            
def _process_row(r, split: str):
    src = Path(r["Image Path"])
    dst = out_root / split / r["label"] / Path(r["Image Index"]).name

    if r["label"] == "dot":
        base_seed = SEED if split == "train" else SEED + 1

        center = choose_random_center(
            img_path=src,
            radius=circle_radius,
            base_seed=base_seed,
            key_str=str(dst.name)
        )

        intensity = compute_intensity(
            img_path=src,
            base_seed=base_seed,
            key_str=str(dst.name)
        )

        add_circle(src, dst, center=center, radius=circle_radius, intensity=intensity)
    else:
        copy_or_link(src, dst)

        
def materialize_split_parallel(dframe: pd.DataFrame, split: str, workers: int = 8, log_every: int = 2000):
    total = len(dframe)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(_process_row, r, split) for _, r in dframe.iterrows()]
        for i, f in enumerate(as_completed(futs), 1):
            # Raise early if any task failed
            _ = f.result()
            if i % log_every == 0 or i == total:
                print(f"[{split}] {i}/{total} ({i/total:.1%})")
                
def label_balanced(dframe: pd.DataFrame, seed: int):
    n = len(dframe)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    half = n // 2
    with_dot = set(idx[:half]) # half with dot, and half with nodot
    dframe["label"] = ["dot" if i in with_dot else "nodot" for i in range(n)]
    return dframe

In [ ]:
# shuffle, split train and test for 80 & 20
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
# split
n_total = len(df)
n_train = int(n_total * 0.8)
train_df = df.iloc[:n_train].copy()
test_df = df.iloc[n_train:].copy()

# balance labels
train_df = label_balanced(train_df,SEED)
test_df = label_balanced(test_df,SEED)

print("Train label counts: \n{}.".format(train_df["label"].value_counts()))
print("\n")
print("Test label counts: \n{}.".format(test_df["label"].value_counts()))

In [ ]:
# create train and test folders
# clean output folders and recreate
shutil.rmtree(out_root, ignore_errors=True)
out_root.mkdir(parents=True, exist_ok=True)

# write files
materialize_split_parallel(train_df, "train")
materialize_split_parallel(test_df, "test")

print("Done with written images to: {}".format(out_root))

In [ ]:
# verify results based on counts and image sizes
def count_pngs(folder: Path):
    return sum(1 for _ in folder.glob("*.png"))

train_dot = out_root / "train" / "dot"
train_nodot = out_root / "train" / "nodot"
test_dot = out_root / "test" / "dot"
test_nodot = out_root / "test" / "nodot"

print("Counts:")
print("Train/dot: {}". format(count_pngs(train_dot)))
print("Train/nodot: {}". format(count_pngs(train_nodot)))
print("Test/dot: {}". format(count_pngs(test_dot)))
print("Test/nodot: {}". format(count_pngs(test_nodot)))

# show the size of the first dot img in train and test and check if present
for i in [train_dot, test_dot]:
    files = list(i.glob("*.png"))
    if files:
        img = Image.open(files[0])
        print("{} first image size (width, height) is: {}".format(i, img.size))
    else:
        print("No images found in {}".format(i))

In [ ]:
dot_images = list((out_root / "train" / "dot").glob("*.png"))
if len(dot_images) == 0:
    raise RuntimeError("No dot images found in train/dot to verify.")

img_path = dot_images[1] if len(dot_images) > 1 else dot_images[0]
# recompute the same center we used when writing (same key_str, same base_seed)
x, y = choose_random_center(
    img_path=img_path,               
    radius=circle_radius,
    base_seed=SEED,                  
    key_str=str(img_path.name)      
)
used_intensity = compute_intensity(
    img_path=img_path,
    base_seed=SEED,
    key_str=str(img_path.name)
)

img = Image.open(img_path).convert("L")
arr = np.array(img)

# check pixel values inside the circle
height, width = arr.shape
yy, xx = np.ogrid[:height, :width]
mask = (xx-x)**2 +(yy-y)**2 <= circle_radius**2
print("Unique values INSIDE circle: ", np.unique(arr[mask]))
print("Expected intensity value: ", used_intensity)

# show image with circle
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,6))

ax1.imshow(arr, cmap="gray")
ax1.set_title("Full Image - Circle r = 1 (random location)")
ax1.axis("off")

ax2.imshow(arr, cmap="gray")
ax2.add_patch(Circle((x,y), circle_radius, fill=False, edgecolor="red",linewidth=2, zorder=2))
ax2.set_title("Full Image - Circle r = 1 marked in red")
ax2.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# add labels to the image generated
root = out_root
LABEL_MAP = {"dot": 1, "nodot" :0} #map dot/nodot as 1 or 0

def make_manifest(split:str, root: Path = root, label_map: dict = LABEL_MAP, pattern: str = "*.png") -> pd.DataFrame:
    # get the path and 0/1 label and return a DataFrame
    rows =[]
    for label_name, label_id in label_map.items():
        folder = root / split / label_name
        if not folder.exists():
            print("Folder not found.")
            continue
        for j in sorted(folder.glob(pattern)):
            rows.append({"path":str(j),"label":int(label_id)})
    return pd.DataFrame(rows, columns=["path","label"])

# build manifests
train_df = make_manifest("train")
test_df = make_manifest("test")

# save manifest to the dataset
train_manifest_path = root / "train_manifest.csv"
test_manifest_path = root / "test_manifest.csv"
train_df.to_csv(train_manifest_path, index=False)
test_df.to_csv(test_manifest_path, index=False)

print("Saved train manifest to {}".format(train_manifest_path))
print("Saved test manifest to {}".format(test_manifest_path))
print("Train rows: ", len(train_df), " | dot: ", (train_df['label']==1).sum(), " | nodot: ", (train_df['label']==0).sum())
print("Test rows: ", len(test_df), " | dot: ", (test_df['label']==1).sum(), " | nodot: ", (test_df['label']==0).sum())
print("\nPreview:\n", train_df.head())

#### Random Circle r = 2, random intensity based on intensity histogram

In [ ]:
# output files
SAMPLE_CSV = WORK_DIR / "meta_NIH-CXR14.csv"
SAMPLE_WITH_PATHS = WORK_DIR / "NIH-CXR14_with_paths.csv"
out_root = WORK_DIR / "random_c2"

In [ ]:
# general settings
SEED = 42
dot_int = 255
WORK_DIR.mkdir(parents=True, exist_ok=True)

# setting circle radius
circle_radius = 2

In [ ]:
# load meta data
df_meta = pd.read_csv(RAW_CSV)
print("Meta data shape: {}".format(df_meta.shape))
print("\n")
print("Colums: {}".format(list(df_meta.columns)))
print("\n")
df_meta.to_csv(SAMPLE_CSV, index=False)
df_meta.head()

In [ ]:
# shuffle, split train and test for 80 & 20
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
# split
n_total = len(df)
n_train = int(n_total * 0.8)
train_df = df.iloc[:n_train].copy()
test_df = df.iloc[n_train:].copy()

# balance labels
train_df = label_balanced(train_df,SEED)
test_df = label_balanced(test_df,SEED)

print("Train label counts: \n{}.".format(train_df["label"].value_counts()))
print("\n")
print("Test label counts: \n{}.".format(test_df["label"].value_counts()))

In [ ]:
# create train and test folders
# clean output folders and recreate
shutil.rmtree(out_root, ignore_errors=True)
out_root.mkdir(parents=True, exist_ok=True)

# write files
materialize_split_parallel(train_df, "train")
materialize_split_parallel(test_df, "test")

print("Done with written images to: {}".format(out_root))

In [ ]:
# verify results based on counts and image sizes
def count_pngs(folder: Path):
    return sum(1 for _ in folder.glob("*.png"))

train_dot = out_root / "train" / "dot"
train_nodot = out_root / "train" / "nodot"
test_dot = out_root / "test" / "dot"
test_nodot = out_root / "test" / "nodot"

print("Counts:")
print("Train/dot: {}". format(count_pngs(train_dot)))
print("Train/nodot: {}". format(count_pngs(train_nodot)))
print("Test/dot: {}". format(count_pngs(test_dot)))
print("Test/nodot: {}". format(count_pngs(test_nodot)))

# show the size of the first dot img in train and test and check if present
for i in [train_dot, test_dot]:
    files = list(i.glob("*.png"))
    if files:
        img = Image.open(files[0])
        print("{} first image size (width, height) is: {}".format(i, img.size))
    else:
        print("No images found in {}".format(i))

In [ ]:
dot_images = list((out_root / "train" / "dot").glob("*.png"))
if len(dot_images) == 0:
    raise RuntimeError("No dot images found in train/dot to verify.")

img_path = dot_images[1] if len(dot_images) > 1 else dot_images[0]
# recompute the same center we used when writing (same key_str, same base_seed)
x, y = choose_random_center(
    img_path=img_path,               
    radius=circle_radius,
    base_seed=SEED,                  
    key_str=str(img_path.name)      
)
used_intensity = compute_intensity(
    img_path=img_path,
    base_seed=SEED,
    key_str=str(img_path.name)
)

img = Image.open(img_path).convert("L")
arr = np.array(img)

# check pixel values inside the circle
height, width = arr.shape
yy, xx = np.ogrid[:height, :width]
mask = (xx-x)**2 +(yy-y)**2 <= circle_radius**2
print("Unique values INSIDE circle: ", np.unique(arr[mask]))
print("Expected intensity value: ", used_intensity)

# show image with circle
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(14,6))

ax1.imshow(arr, cmap="gray")
ax1.set_title("Full Image - Circle r = 2 (random location)")
ax1.axis("off")

ax2.imshow(arr, cmap="gray")
ax2.add_patch(Circle((x,y), circle_radius, fill=False, edgecolor="red",linewidth=2, zorder=2))
ax2.set_title("Full Image - Circle r = 2 marked in red")
ax2.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# add labels to the image generated
root = out_root
LABEL_MAP = {"dot": 1, "nodot" :0} #map dot/nodot as 1 or 0

def make_manifest(split:str, root: Path = root, label_map: dict = LABEL_MAP, pattern: str = "*.png") -> pd.DataFrame:
    # get the path and 0/1 label and return a DataFrame
    rows =[]
    for label_name, label_id in label_map.items():
        folder = root / split / label_name
        if not folder.exists():
            print("Folder not found.")
            continue
        for j in sorted(folder.glob(pattern)):
            rows.append({"path":str(j),"label":int(label_id)})
    return pd.DataFrame(rows, columns=["path","label"])

# build manifests
train_df = make_manifest("train")
test_df = make_manifest("test")

# save manifest to the dataset
train_manifest_path = root / "train_manifest.csv"
test_manifest_path = root / "test_manifest.csv"
train_df.to_csv(train_manifest_path, index=False)
test_df.to_csv(test_manifest_path, index=False)

print("Saved train manifest to {}".format(train_manifest_path))
print("Saved test manifest to {}".format(test_manifest_path))
print("Train rows: ", len(train_df), " | dot: ", (train_df['label']==1).sum(), " | nodot: ", (train_df['label']==0).sum())
print("Test rows: ", len(test_df), " | dot: ", (test_df['label']==1).sum(), " | nodot: ", (test_df['label']==0).sum())
print("\nPreview:\n", train_df.head())